# Predict the gene-enhancer pairs based on correlation of transcription in Beas2B cells
This code is based on Ru's code at https://github.com/Dowell-Lab/bidir_gene_pairs/blob/main/R/bidir_gene_correlations_allsamples.R (used September 8th, 2025)

In [1]:
library(data.table)

## 1. Read in Counts (fixed counts)

In [2]:
input_dir = "/scratch/Users/hoto7260/nextflow_out/Bidir_Count/"
td_10 <- fread(paste0(input_dir, "CUWA/Sasse2019B2B_10min/counts/fixed_MU_nongenetss_Sasse2019B2B_9.8.25_counts.txt"))
td_10_genes <- fread(paste0(input_dir, "CUWA/Sasse2019B2B_10min/counts/fixed_genes_Sasse2019B2B_9.8.25_counts.txt"))

td_30 <- fread(paste0(input_dir, "CUWA/Sasse2019B2B_30_min/counts/fixed_MU_nongenetss_Sasse2019B2B_9.8.25_counts.txt"))
td_30_genes <- fread(paste0(input_dir, "CUWA/Sasse2019B2B_30_min/counts/fixed_genes_Sasse2019B2B_9.8.25_counts.txt"))

gally <- fread(paste0(input_dir, "CUWA/Gally2020B2B/counts/fixed_MU_nongenetss_Gally2020B2B_9.8.25_counts.txt"))
gally_genes <- fread(paste0(input_dir, "CUWA/Gally2020B2B/counts/fixed_genes_Gally2020B2B_9.8.25_counts.txt"))

all_tREs = union(union(td_10$Geneid, td_30$Geneid), gally$Geneid)
length(all_tREs)


[1] 57503

In [3]:
# get the WSP and UPM ones too
upm <- fread("/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/UPM/counts/fixed_full_bid_CUWA_UPM_3.26.25_counts.txt")
upm_genes <- fread("/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/UPM/counts/fixed_genes_CUWA_UPM_3.26.25_counts.txt")
wsp <- fread("/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/WSP/counts/fixed_full_bid_CUWA_WSP_3.26.25_counts.txt")
wsp_genes <- fread("/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/WSP/counts/fixed_genes_CUWA_WSP_3.26.25_counts.txt")
adp <- fread("/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/ADP/counts/fixed_full_bid_CUWA_ADP_3.26.25_counts.txt")
adp_genes <- fread("/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/ADP/counts/fixed_genes_CUWA_ADP_3.26.25_counts.txt")

nontss_regions_wsp <- fread("/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/WSP/regions/nontss_bid_CUWA_WSP_3.26.25_forTFEA.bed")
nontss_regions_upm <- fread("/scratch/Users/hoto7260/Resp_Env/Comb_UPM_WSP_ADP/UPM/regions/nontss_bid_CUWA_UPM_3.26.25_forTFEA.bed")
# remove TSS bidirectionals

nontss_regions <- union(nontss_regions_wsp$V4, nontss_regions_upm$V4)

upm = upm[upm$Geneid %in% union(nontss_regions, all_tREs),]
wsp = wsp[wsp$Geneid %in% union(nontss_regions, all_tREs),]
adp = adp[adp$Geneid %in% union(nontss_regions, all_tREs),]


In [5]:
dim(td_10)
dim(td_30)
dim(gally)
dim(upm)
dim(wsp)
dim(adp)

[1] 54102    15

[1] 55015    13

[1] 57420     8

[1] 53531    12

[1] 54065    12

[1] 55119    10

In [47]:
gally[1:2,]
gally_genes[1:2,]

Geneid,Chr,Start,End,Strand,Length,./SRR12482692.mmfilt.sorted.bam,./SRR12482693.mmfilt.sorted.bam
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>
chr1:785582-786076,chr1;chr1,784855;785829,785829;786829,-;+,1976,25,69
chr1:805310-805780,chr1;chr1,804753;805545,805545;806545,-;+,1794,16,39


Geneid,Chr,Start,End,Strand,Length,./SRR12482692.mmfilt.sorted.bam,./SRR12482693.mmfilt.sorted.bam
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>
GLIDR:NR_126044.1,chr9,39807687,39808385,-,699,2,14
TIA1:NM_022037.4,chr2;chr2,70209443;70242652,70240652;70247878,-;-,36437,2286,4564


### Make a gene+tRE bidirectional matrix

In [6]:
# only consider the long isoforms of genes to get the TPMs
long_isoforms <- fread("/scratch/Shares/dowell/genomes/hg38/ncbi/hg38_refseq_longisof_transcripts_counting.bed")
long_isoforms[1:2,]

td_10_genes_long <- td_10_genes[td_10_genes$Geneid %in% long_isoforms$V4,]
td_30_genes_long <- td_30_genes[td_30_genes$Geneid %in% long_isoforms$V4,]
gally_genes_long <- gally_genes[gally_genes$Geneid %in% long_isoforms$V4,]
upm_genes_long <- upm_genes[upm_genes$Geneid %in% long_isoforms$V4,]
wsp_genes_long <- wsp_genes[wsp_genes$Geneid %in% long_isoforms$V4,]
adp_genes_long <- adp_genes[adp_genes$Geneid %in% long_isoforms$V4,]

dim(td_10_genes_long)
dim(td_30_genes_long)
dim(gally_genes_long)
dim(upm_genes_long)
dim(wsp_genes_long)
dim(adp_genes_long)

td_10_long <- rbind(td_10, td_10_genes_long)
td_30_long <- rbind(td_30, td_30_genes_long)
gally_long <- rbind(gally, gally_genes_long)
upm_long <- rbind(upm, upm_genes_long)
wsp_long <- rbind(wsp, wsp_genes_long)
adp_long <- rbind(adp, adp_genes_long)


V1,V2,V3,V4,V5,V6
<chr>,<int>,<int>,<chr>,<chr>,<chr>
chr1,12623,14409,DDX11L1:NR_046018.2,.,+
chr1,14361,28620,WASH7P:NR_024540.1,.,-


[1] 28802    15

[1] 28820    13

[1] 28789     8

[1] 28774    12

[1] 28799    12

[1] 28813    10

In [7]:
td_10 <- rbind(td_10, td_10_genes)
td_30 <- rbind(td_30, td_30_genes)
gally <- rbind(gally, gally_genes)
upm <- rbind(upm, upm_genes)
wsp <- rbind(wsp, wsp_genes)
adp <- rbind(adp, adp_genes)

dim(td_10)
dim(td_30)
dim(gally)
dim(upm)
dim(wsp)
dim(adp)

[1] 96234    15

[1] 97169    13

[1] 99539     8

[1] 95630    12

[1] 96195    12

[1] 97263    10

In [8]:
library(dplyr)
# calculate TPM
calculate_tpm <- function(df, long_df,
                          meta_cols = c("Geneid","Chr","Start","End","Strand","Length")) {
    # change into dataframes
 df <- as.data.frame(df)
    long_df <- as.data.frame(long_df)
  # only keep one chromosome (currenlty split for GTFs)
    df$Chr <- sub(";.*", "", df$Chr)
    df$Strand <- sub(";.*", "", df$Strand)

    # have all bidirectionals have the same chromosome
    bids = df$Geneid[grep("chr", df$Geneid)]
    df[df$Geneid %in% bids,]$Strand <- "."
  # Get the min start and max end for linking
    df <- df %>%
  mutate(
    Start = sapply(strsplit(Start, ";"), function(x) min(as.numeric(x))),
    End   = sapply(strsplit(End, ";"),   function(x) max(as.numeric(x)))
  )
  # basic checks
  if (!"Length" %in% names(df) || !"Length" %in% names(long_df)) {
    stop("Both 'df' and 'long_df' must contain a 'Length' column.")
  }

  # identify count columns (exclude metadata)
  counts_df   <- setdiff(names(df), meta_cols)
  counts_long <- setdiff(names(long_df), meta_cols)
  common      <- intersect(counts_df, counts_long)
  if (length(common) == 0) stop("No common count columns found between df and long_df.")
  if (length(common) < length(counts_df)) {
    message("Using only common count columns: ", paste(common, collapse = ", "))
  }

  # prepare numeric matrices (rows = genes, cols = samples)
  long_mat <- apply(as.matrix(long_df[common]), 2, as.numeric)
  df_mat   <- apply(as.matrix(df[common]), 2, as.numeric)
  # get length vectors
  long_len <- long_df$Length 
  df_len   <- df$Length 

  # warn if zero length present
  if (any(long_len == 0, na.rm = TRUE)) warning("Zero length(s) found in long_df -> produces Inf/NaN in RPK.")
  if (any(df_len   == 0, na.rm = TRUE)) warning("Zero length(s) found in df -> produces Inf/NaN in RPK.")

  # compute RPK: divide each row by its length (vectorized)
  # (sweep with MARGIN = 1 divides each row by corresponding length)
  long_rpk <- sweep(long_mat, 1, long_len, FUN = "/")
  df_rpk   <- sweep(df_mat,   1, df_len,   FUN = "/")


  # column sums of RPK from long_df -> the denominators (size factors)
  size_factors <- colSums(long_rpk, na.rm = TRUE)

  if (any(size_factors == 0, na.rm = TRUE)) {
    warning("One or more size factors (colSums of long_rpk) are zero; resulting TPMs will be NA/Inf for those samples.")
  }

  # compute TPM for df using the size_factors from long_df
  tpm_mat <- sweep(df_rpk, 2, size_factors, FUN = "/") * 1e6

  # set column names and assemble result
  colnames(tpm_mat) <- paste0(colnames(tpm_mat), "_TPM")
  result <- data.frame(Geneid = df$Geneid, chrom = df$Chr, start = df$Start, 
                       stop = df$End, strand = df$Strand, as.data.frame(tpm_mat), check.names = FALSE)
  

  return(result)
}


# checking it works as expected
df <- data.frame(
  Geneid = c("g1","chr2","g3"),
  Chr = "chr1", Start = c("1;1", "2;2", "3;3"), 
    End = c("11;11", "12;12", "13;13"), Strand = "+",
  Length = c(1000, 2000, 1500),
  SampleA = c(100, 50, 25),
  SampleB = c(200, 25, 125)
)


# SampleA
#100/1000 = 0.1
#50/2000 = 0.025
#25/1500 = 0.01666667

#Sample B
#200/1000 = .2
#25/2000 = 0.0125
#125/1500 = 0.08333

#Only using 1 and 3 for colSums —> SampleA:0.1166667, SampleB: 0.28333

#SampleA
#TPM = 0.1/0.11116666 * 1e6 = 857142
#TPM = 0.025/0.11116666 * 1e6 = 214285
#TPM = 0.0166667/0.1166667 *1e6 = 142857
#SampleB
#TPM = 0.2/0.28333 * 1e6 = 705890
#TPM = 0.0125/0.28333 * 1e6 = 44117
#TPM = 0.08333/0.28333 * 1e6 = 294117


# long_df is a subset (long isoforms only)
long_df <- df[c(1,3), ]  # use g1 and g3 as "long" isoforms
long_df
calculate_tpm(df, long_df)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




,Geneid,Chr,Start,End,Strand,Length,SampleA,SampleB
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,g1,chr1,1;1,11;11,+,1000,100,200
3,g3,chr1,3;3,13;13,+,1500,25,125


Geneid,chrom,start,stop,strand,SampleA_TPM,SampleB_TPM
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>
g1,chr1,1,11,+,857142.9,705882.35
chr2,chr1,2,12,.,214285.7,44117.65
g3,chr1,3,13,+,142857.1,294117.65


In [10]:
td_10_tpm <- calculate_tpm(td_10, td_10_long)
td_30_tpm <- calculate_tpm(td_30, td_30_long)
gally_tpm <- calculate_tpm(gally, gally_long)
upm_tpm <- calculate_tpm(upm, upm_long)
wsp_tpm <- calculate_tpm(wsp, wsp_long)
adp_tpm <- calculate_tpm(adp, adp_long)
td_10[1:2,]
td_10_tpm[1:2,]
td_30[1:2,]
td_30_tpm[1:2,]
gally[1:2,]
gally_tpm[1:2,]
upm[1:2,]
upm_tpm[1:2,]
wsp[1:2,]
wsp_tpm[1:2,]
adp[1:2,]
adp_tpm[1:2,]

Geneid,Chr,Start,End,Strand,Length,./SRR8429046.mmfilt.sorted.bam,./SRR8429047.mmfilt.sorted.bam,./SRR8429048.mmfilt.sorted.bam,./SRR8429049.mmfilt.sorted.bam,./SRR8429050.mmfilt.sorted.bam,./SRR8429051.mmfilt.sorted.bam,./SRR8429052.mmfilt.sorted.bam,./SRR8429053.mmfilt.sorted.bam,./SRR8429054.mmfilt.sorted.bam
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr1:1133229-1133779,chr1;chr1,1132693;1133504,1133504;1134504,-;+,1813,1,1,0,0,0,0,1,23,1
chr1:1353573-1354013,chr1;chr1,1353105;1353793,1353793;1354793,-;+,1690,319,216,484,333,337,294,312,84,46


,Geneid,chrom,start,stop,strand,./SRR8429046.mmfilt.sorted.bam_TPM,./SRR8429047.mmfilt.sorted.bam_TPM,./SRR8429048.mmfilt.sorted.bam_TPM,./SRR8429049.mmfilt.sorted.bam_TPM,./SRR8429050.mmfilt.sorted.bam_TPM,./SRR8429051.mmfilt.sorted.bam_TPM,./SRR8429052.mmfilt.sorted.bam_TPM,./SRR8429053.mmfilt.sorted.bam_TPM,./SRR8429054.mmfilt.sorted.bam_TPM
,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1:1133229-1133779,chr1,1132693,1134504,.,0.07606955,0.08708922,0.00000,0.00000,0.00000,0.00000,0.07157696,3.288911,0.1207089
2,chr1:1353573-1354013,chr1,1353105,1354793,.,26.03230700,20.18037596,43.28361,25.80353,28.18709,25.61123,23.95736005,12.885897,5.9567357


Geneid,Chr,Start,End,Strand,Length,./SRR8429055.mmfilt.sorted.bam,./SRR8429056.mmfilt.sorted.bam,./SRR8429057.mmfilt.sorted.bam,./SRR8429058.mmfilt.sorted.bam,./SRR8429059.mmfilt.sorted.bam,./SRR8429060.mmfilt.sorted.bam,./SRR8429061.mmfilt.sorted.bam
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr1:1133229-1133779,chr1;chr1,1132693;1133504,1133504;1134504,-;+,1813,0,0,0,0,0,0,0
chr1:1353573-1354013,chr1;chr1,1353105;1353793,1353793;1354793,-;+,1690,35,17,39,42,69,10,5


,Geneid,chrom,start,stop,strand,./SRR8429055.mmfilt.sorted.bam_TPM,./SRR8429056.mmfilt.sorted.bam_TPM,./SRR8429057.mmfilt.sorted.bam_TPM,./SRR8429058.mmfilt.sorted.bam_TPM,./SRR8429059.mmfilt.sorted.bam_TPM,./SRR8429060.mmfilt.sorted.bam_TPM,./SRR8429061.mmfilt.sorted.bam_TPM
,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1:1133229-1133779,chr1,1132693,1134504,.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,chr1:1353573-1354013,chr1,1353105,1354793,.,4.674117,1.792095,3.886025,4.573681,6.063342,1.008037,1.446679


Geneid,Chr,Start,End,Strand,Length,./SRR12482692.mmfilt.sorted.bam,./SRR12482693.mmfilt.sorted.bam
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>
chr1:785582-786076,chr1;chr1,784855;785829,785829;786829,-;+,1976,25,69
chr1:805310-805780,chr1;chr1,804753;805545,805545;806545,-;+,1794,16,39


...

,Geneid,chrom,start,stop,strand,./mmfiltbams/sm36-ALI-D21_120UPM-1.mmfilt.sorted.bam_TPM,./mmfiltbams/sm36-ALI-D21_120UPM-2.mmfilt.sorted.bam_TPM,./mmfiltbams/sm36-ALI-D21_30UPM-1.mmfilt.sorted.bam_TPM,./mmfiltbams/sm36-ALI-D21_30UPM-2.mmfilt.sorted.bam_TPM,./mmfiltbams/sm36-ALI-D21_veh-1.mmfilt.sorted.bam_TPM,./mmfiltbams/sm36-ALI-D21_veh-2.mmfilt.sorted.bam_TPM
,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1:1133229-1133779,chr1,1132693,1134504,.,15.96459,17.47240,16.67989207,18.670929,21.41107,24.223381
2,chr1:1353573-1354013,chr1,1353105,1354793,.,10.13843,11.38166,0.02933422,1.546081,7.07517,6.416655


Geneid,Chr,Start,End,Strand,Length,./mmfiltbams/SRR13772348.mmfilt.sorted.bam,./mmfiltbams/SRR13772349.mmfilt.sorted.bam,./mmfiltbams/SRR13772350.mmfilt.sorted.bam,./mmfiltbams/SRR13772351.mmfilt.sorted.bam,./mmfiltbams/SRR13772352.mmfilt.sorted.bam,./mmfiltbams/SRR13772353.mmfilt.sorted.bam
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr1:785582-786076,chr1;chr1,784855;785829,785829;786829,-;+,1976,14,19,25,32,63,69
chr1:805310-805780,chr1;chr1,804753;805545,805545;806545,-;+,1794,15,7,20,19,23,42


,Geneid,chrom,start,stop,strand,./mmfiltbams/SRR13772348.mmfilt.sorted.bam_TPM,./mmfiltbams/SRR13772349.mmfilt.sorted.bam_TPM,./mmfiltbams/SRR13772350.mmfilt.sorted.bam_TPM,./mmfiltbams/SRR13772351.mmfilt.sorted.bam_TPM,./mmfiltbams/SRR13772352.mmfilt.sorted.bam_TPM,./mmfiltbams/SRR13772353.mmfilt.sorted.bam_TPM
,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1:785582-786076,chr1,784855,786829,.,0.6288437,0.8579477,1.0674170,1.3135652,2.994304,2.977500
2,chr1:805310-805780,chr1,804753,806545,.,0.7421137,0.3481527,0.9405645,0.8590526,1.204059,1.996257


Geneid,Chr,Start,End,Strand,Length,./SRR18838287.mmfilt.sorted.bam,./SRR18838288.mmfilt.sorted.bam,./SRR18838289.mmfilt.sorted.bam,./SRR18838290.mmfilt.sorted.bam
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>
chr1:785582-786076,chr1;chr1,784855;785829,785829;786829,-;+,1976,15,4,41,9
chr1:805310-805780,chr1;chr1,804753;805545,805545;806545,-;+,1794,13,2,17,8


,Geneid,chrom,start,stop,strand,./SRR18838287.mmfilt.sorted.bam_TPM,./SRR18838288.mmfilt.sorted.bam_TPM,./SRR18838289.mmfilt.sorted.bam_TPM,./SRR18838290.mmfilt.sorted.bam_TPM
,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,chr1:785582-786076,chr1,784855,786829,.,1.174133,0.6020762,3.005157,0.9876785
2,chr1:805310-805780,chr1,804753,806545,.,1.120815,0.3315782,1.372451,0.9670025


In [11]:
# combine by Geneid with any cases where Geneid not found become NA

b2b_dfs <- list(td_10_tpm, td_30_tpm, gally_tpm, wsp_tpm)
sm_dfs <- list(upm_tpm, adp_tpm)
all_dfs <- list(td_10_tpm, td_30_tpm, gally_tpm, wsp_tpm, upm_tpm, adp_tpm)

# merge them all by Geneid
tpm_b2b <- Reduce(function(x, y) full_join(x, y, by = c("Geneid", "chrom", "start", "stop", "strand")), b2b_dfs)
tpm_sm <- Reduce(function(x, y) full_join(x, y, by = c("Geneid", "chrom", "start", "stop", "strand")), sm_dfs)                  
tpm_all <- Reduce(function(x, y) full_join(x, y, by = c("Geneid", "chrom", "start", "stop", "strand")), all_dfs)

In [12]:
# Collapse duplicates by Geneid + chrom + strand
# pick the lowest start and longest stop since those can change
tpm_b2b <- tpm_b2b %>%
  group_by(Geneid, chrom, strand) %>%
  summarise(
    start = min(start, na.rm = TRUE),
    stop  = max(stop, na.rm = TRUE),
    across(where(is.numeric), ~ sum(.x, na.rm = TRUE)),
    .groups = "drop"
  )

tpm_sm <- tpm_sm %>%
  group_by(Geneid, chrom, strand) %>%
  summarise(
    start = min(start, na.rm = TRUE),
    stop  = max(stop, na.rm = TRUE),
    across(where(is.numeric), ~ sum(.x, na.rm = TRUE)),
    .groups = "drop"
  )

tpm_all <- tpm_all %>%
  group_by(Geneid, chrom, strand) %>%
  summarise(
    start = min(start, na.rm = TRUE),
    stop  = max(stop, na.rm = TRUE),
    across(where(is.numeric), ~ sum(.x, na.rm = TRUE)),
    .groups = "drop"
  )

In [13]:
# check no duplicates
length(tpm_all$Geneid[duplicated(tpm_all$Geneid)])
length(tpm_b2b$Geneid[duplicated(tpm_b2b$Geneid)])
length(tpm_sm$Geneid[duplicated(tpm_sm$Geneid)])

[1] 0

[1] 0

[1] 0

In [26]:
write.table(tpm_all, "TPM_B2BsmAEC_tREGene.txt", quote=FALSE, row.names=FALSE)
write.table(tpm_b2b, "TPM_B2B_tREGene.txt", quote=FALSE, row.names=FALSE)
write.table(tpm_sm, "TPM_smAEC_tREGene.txt", quote=FALSE, row.names=FALSE)

## Run the get_bidir_gene_pairs.sbatch and .sh

In [27]:
tpm_b2b <- fread("TPM_B2B_tREGene.txt")
tpm_sm <- fread("TPM_smAEC_tREGene.txt")
tpm_all <- fread("TPM_B2BsmAEC_tREGene.txt")

## Get the top calls per enhancer and per gene for each category and save
* Also get the stats of each of those calls

Two dataframes, one where rows are enhancers and other where rows are genes
* 5 columns with () separating out the values from All, B2B, and smAEC
* 5 columns correspond to top choice from closest to TES, closest to TSS, All, B2B, smAEC
* Top_All: Geneid (PCC,pval,%), (PCC,pval,%), (PCC,pval,%), Top_B2B: Geneid (PCC,pval,%), (PCC,pval,%), (PCC,pval,%), Top_smAEC: Geneid (PCC,pval,%), (PCC,pval,%), (PCC,pval,%)

In [30]:
#"pcc", "adj_p_BH", "nObs", "distance_tss", "distance_tes", "position", "percent_transcribed_both"

out_path = "/scratch/Users/hoto7260/Resp_Env/All_B2B/bidir_gene_pairs/"

gene_links_list <- list()
bid_links_list  <- list()

for (chrom_num in c("chr1", "chr2", "chr3", "chr4", "chr5", "chr6", "chr7", 
                    "chr8", "chr9", "chr10", "chr11", "chr12", "chr13", "chr14", 
                    "chr15", "chr16", "chr17", "chr18", "chr19", "chr20", 
                    "chr21", "chr22", "chrX", "chrY")) {
    filt_tpm = tpm_all[tpm_all$chrom == chrom_num,]
    genes <- unique(filt_tpm$Geneid[grep("_", filt_tpm$Geneid)])
    bids <- unique(filt_tpm$Geneid[grep("chr", filt_tpm$Geneid)])
    b2b_links <- fread(paste0(out_path, "pearson_correlation_", chrom_num, "_Beas2B.tsv.gz"))
    sm_links <- fread(paste0(out_path, "pearson_correlation_", chrom_num, "_smAEC.tsv.gz"))
    all_links <- fread(paste0(out_path, "pearson_correlation_", chrom_num, "_Beas2BsmAEC.tsv.gz"))

    # get the best predicted linkages based first on p-value then percentage
    gene_links_list[[chrom_num]] <- rbindlist(
        lapply(genes, function(x) as.list(get_top_link_values(x, b2b_links, sm_links, all_links, gene = TRUE)))
    )
    
    bid_links_list[[chrom_num]] <- rbindlist(
        lapply(bids, function(x) as.list(get_top_link_values(x, b2b_links, sm_links, all_links, gene = FALSE)))
    )
                                      
                                      
# combine into final big tables
gene_links <- rbindlist(gene_links_list, fill = TRUE)
bid_links  <- rbindlist(bid_links_list, fill = TRUE)  
        }

In [31]:
dim(gene_links)
length(unique(gene_links$Feature))
dim(bid_links)
length(unique(bid_links$Feature))
nrow(gene_links) + nrow(bid_links)
length(unique(gene_links$Feature)) + length(unique(bid_links$Feature))
dim(tpm_all)
length(unique(tpm_all$Geneid))

[1] 42180     6

[1] 42180

[1] 57958     6

[1] 57958

[1] 100138

[1] 100138

[1] 100138     39

[1] 100138

In [32]:
# save files
write.table(gene_links, "Gene_toptRE_links_9.11.25.txt", quote=FALSE, row.names=FALSE, sep="\t")
write.table(bid_links, "TRE_topGene_links_9.11.25.txt", quote=FALSE, row.names=FALSE, sep="\t")

### Try to link bidirectionals naming to that in annotations for SNPs

In [ ]:
# get the mu based names
bids <- tpm_all$Geneid[grep("chr", tpm_all$Geneid)]
split <- stringr::str_split_fixed(bids, ":", 2)
split2 <- stringr::str_split_fixed(split[,2], "-", 2)
mus <- as.integer((as.numeric(split2[,1]) + as.numeric(split2[,2]))/2)
bid_names <- paste0(split[,1], ":", mus)

In [33]:
bid_links[1:2,]
split <- stringr::str_split_fixed(bid_links$Feature, ":", 2)
split2 <- stringr::str_split_fixed(split[,2], "-", 2)
bid_links$chr <- split[,1]
bid_links$start <- as.numeric(split2[,1])
bid_links$end <- as.numeric(split2[,2])
mus <- as.integer((bid_links$start + bid_links$end)/2)
bid_links$name <- paste0(split[,1], ":", mus)
bid_links[1:2,]

Feature,AllCorr,B2BCorr,smAECCorr,DistTSS,DistTES
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
chr1:100028659-100028973,"SLC35A3:NM_001271685.2(0.46,0.06,67%), (0.43,0.28,59%), (0.34,0.6,60%)","LINC01349:NR_038914.1(0.45,0.12,54%), (0.67,0.04,59%), (0.12,0.94,27%)","SLC35A3:NM_001271685.2(0.46,0.06,67%), (0.43,0.28,59%), (0.34,0.6,60%)","MFSD14A:NM_033055.3(0.31,0.26,67%), (0.31,0.5,59%), (0.2,0.78,60%)","SLC35A3:NM_001271685.2(0.46,0.06,67%), (0.43,0.28,59%), (0.34,0.6,60%)"
chr1:100031284-100031596,"SLC35A3:NM_012243.3(0.69,0,62%), (0.62,0.12,48%), (0.02,0.98,67%)","FRRS1:NM_001013660.4(0.67,0,62%), (0.73,0.04,48%), (0.55,0.28,67%)","DPH5-DT:NR_109849.1(0.44,0.1,62%), (0.36,0.47,48%), (0.65,0.17,67%)","MFSD14A:NM_033055.3(0.53,0.04,62%), (0.7,0.06,48%), (0.21,0.75,67%)","SLC35A3:NM_012243.3(0.69,0,62%), (0.62,0.12,48%), (0.02,0.98,67%)"


Feature,AllCorr,B2BCorr,smAECCorr,DistTSS,DistTES,chr,start,end,name
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
chr1:100028659-100028973,"SLC35A3:NM_001271685.2(0.46,0.06,67%), (0.43,0.28,59%), (0.34,0.6,60%)","LINC01349:NR_038914.1(0.45,0.12,54%), (0.67,0.04,59%), (0.12,0.94,27%)","SLC35A3:NM_001271685.2(0.46,0.06,67%), (0.43,0.28,59%), (0.34,0.6,60%)","MFSD14A:NM_033055.3(0.31,0.26,67%), (0.31,0.5,59%), (0.2,0.78,60%)","SLC35A3:NM_001271685.2(0.46,0.06,67%), (0.43,0.28,59%), (0.34,0.6,60%)",chr1,100028659,100028973,chr1:100028816
chr1:100031284-100031596,"SLC35A3:NM_012243.3(0.69,0,62%), (0.62,0.12,48%), (0.02,0.98,67%)","FRRS1:NM_001013660.4(0.67,0,62%), (0.73,0.04,48%), (0.55,0.28,67%)","DPH5-DT:NR_109849.1(0.44,0.1,62%), (0.36,0.47,48%), (0.65,0.17,67%)","MFSD14A:NM_033055.3(0.53,0.04,62%), (0.7,0.06,48%), (0.21,0.75,67%)","SLC35A3:NM_012243.3(0.69,0,62%), (0.62,0.12,48%), (0.02,0.98,67%)",chr1,100031284,100031596,chr1:100031440


In [126]:
bid_links[bid_links$chr == "chr1" & bid_links$start < 6249500 & bid_links$end > 6249100,]

bid_links$Feature[grep("chr1:624", bid_links$Feature)]

length(unique(bid_links$Feature))

Feature,AllCorr,B2BCorr,smAECCorr,DistTSS,DistTES,name,start,end,chr
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>


[1] "chr1:62446327-62446703" "chr1:62450327-62450677" "chr1:62458531-62459081"
[4] "chr1:62461129-62461659" "chr1:62462536-62463024" "chr1:62464073-62464541"
[7] "chr1:62465327-62465871"

[1] 57515

In [18]:
name[1:2]

[1] "chr10:100006622" "chr10:100044429"

In [25]:
## ALL MISSING ITEMS ARE GENE TSS BIDS AND THEREFORE HAVE AUTOMATIC LINKAGE
bids <- tpm_all$Geneid[grep("chr", tpm_all$Geneid)]
split <- stringr::str_split_fixed(bids, ":", 2)
split2 <- stringr::str_split_fixed(split[,2], "-", 2)
chr <- split[,1]
start <- as.numeric(split2[,1])
end <- as.numeric(split2[,2])
mus <- as.integer((start + end)/2)
name <- paste0(split[,1], ":", mus)
snps$name[1:2]
missing = setdiff(snps$central, name)
length(missing)
missing_split = missing[grepl(";", missing)]
length(missing_split)
missing_nosplit = setdiff(missing, missing_split)
length(missing_nosplit)
snps_missing = snps[snps$central %in% missing_nosplit,]
table(snps_missing$note)

[1] "chr1:966814-968936" "chr1:966814-968936"

[1] 1157

[1] 990

[1] 167


Gene TSS Bid 
         174 

In [16]:
# link back to snp based names
snps <- readxl::read_excel("aou.copd_asthma_UPM_WSP.xlsx")

snps$name <-paste0("chr", snps$CHR, ":", snps$start, "-", snps$stop)
nrow(snps)
snps[1:2,]
bid_names[1:4]
length(intersect(tpm_all$Geneid, snps$name))
length(intersect(bid_links$name, snps$central))


missing = setdiff(snps$central, bid_links$name)
length(missing)
length(missing[grep(";", missing)])
missing_no_split = setdiff(missing, missing[grep(";", missing)])
snps[snps$central %in% missing_no_split,]

[1] 6164

CHR,start,stop,central,note,BP,rsID,OR_copd,OR_asthma,P_copd,P_asthma,meta_p_weighted,same_direction,name
<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,966814,968936,chr1:968123,LIET Gene TSS Intron,967369,rs113967711,0.9846,0.9606,0.6414,0.02325,0.11765640,Yes,chr1:966814-968936
1,966814,968936,chr1:968123,LIET Gene TSS Intron,967941,rs6669800,1.0450,0.9737,0.1586,0.11470,0.05940026,No,chr1:966814-968936


ERROR: Error in eval(expr, envir, enclos): object 'bid_names' not found


In [29]:

get_top_link_values <- function(feat_use, b2b_links, sm_links, all_links, gene=TRUE) {
    if (gene) {
        filt_b2b_links = b2b_links[b2b_links$transcript_1 == feat_use,]
        filt_sm_links = sm_links[sm_links$transcript_1 == feat_use,]
        filt_all_links = all_links[all_links$transcript_1 == feat_use,]
        # order first by adj_p_BH then percent
        setorder(filt_b2b_links, adj_p_BH, -percent_transcribed_both)
        setorder(filt_sm_links, adj_p_BH, -percent_transcribed_both)
        setorder(filt_all_links, adj_p_BH, -percent_transcribed_both)
        # get the top options
        top_b2b = filt_b2b_links$transcript_2[1]
        top_sm = filt_sm_links$transcript_2[1]
        top_all = filt_all_links$transcript_2[1]
        top_disttss = filt_all_links[order(abs(filt_all_links$distance_tss)),]$transcript_2[1]
        top_disttes = filt_all_links[order(abs(filt_all_links$distance_tes)),]$transcript_2[1]
        
        b2b_topall <- filt_b2b_links[filt_b2b_links$transcript_2 == top_all,]
        sm_topall <- filt_sm_links[filt_sm_links$transcript_2 == top_all,]
        
        all_topb2b <- filt_all_links[filt_all_links$transcript_2 == top_b2b,]
        sm_topb2b <- filt_sm_links[filt_sm_links$transcript_2 == top_b2b,]
        
        all_topsm <- filt_all_links[filt_all_links$transcript_2 == top_sm,]
        b2b_topsm <- filt_b2b_links[filt_b2b_links$transcript_2 == top_sm,]
        
        all_topdisttss <- filt_all_links[filt_all_links$transcript_2 == top_disttss,]
        b2b_topdisttss <- filt_b2b_links[filt_b2b_links$transcript_2 == top_disttss,]
        sm_topdisttss <- filt_sm_links[filt_sm_links$transcript_2 == top_disttss,]
        
        all_topdisttes <- filt_all_links[filt_all_links$transcript_2 == top_disttes,]
        b2b_topdisttes <- filt_b2b_links[filt_b2b_links$transcript_2 == top_disttes,]
        sm_topdisttes <- filt_sm_links[filt_sm_links$transcript_2 == top_disttes,]
        } else {
        filt_b2b_links = b2b_links[b2b_links$transcript_2 == feat_use,]
        filt_sm_links = sm_links[sm_links$transcript_2 == feat_use,]
        filt_all_links = all_links[all_links$transcript_2 == feat_use,]
        # order first by adj_p_BH then percent
        setorder(filt_b2b_links, adj_p_BH, -percent_transcribed_both)
        setorder(filt_sm_links, adj_p_BH, -percent_transcribed_both)
        setorder(filt_all_links, adj_p_BH, -percent_transcribed_both)
        # get the top options
        top_b2b = filt_b2b_links$transcript_1[1]
        top_sm = filt_sm_links$transcript_1[1]
        top_all = filt_all_links$transcript_1[1]
        top_disttss = filt_all_links[order(abs(filt_all_links$distance_tss)),]$transcript_1[1]
        top_disttes = filt_all_links[order(abs(filt_all_links$distance_tes)),]$transcript_1[1]
        
        b2b_topall <- filt_b2b_links[filt_b2b_links$transcript_1 == top_all,]
        sm_topall <- filt_sm_links[filt_sm_links$transcript_1 == top_all,]
        
        all_topb2b <- filt_all_links[filt_all_links$transcript_1 == top_b2b,]
        sm_topb2b <- filt_sm_links[filt_sm_links$transcript_1 == top_b2b,]
        
        all_topsm <- filt_all_links[filt_all_links$transcript_1 == top_sm,]
        b2b_topsm <- filt_b2b_links[filt_b2b_links$transcript_1 == top_sm,]
        
        all_topdisttss <- filt_all_links[filt_all_links$transcript_1 == top_disttss,]
        b2b_topdisttss <- filt_b2b_links[filt_b2b_links$transcript_1 == top_disttss,]
        sm_topdisttss <- filt_sm_links[filt_sm_links$transcript_1 == top_disttss,]
        
        all_topdisttes <- filt_all_links[filt_all_links$transcript_1 == top_disttes,]
        b2b_topdisttes <- filt_b2b_links[filt_b2b_links$transcript_1 == top_disttes,]
        sm_topdisttes <- filt_sm_links[filt_sm_links$transcript_1 == top_disttes,]
        }
    
    
    # get the stats for each of these
    top_all_value = paste0(top_all, "(", round(filt_all_links$pcc[1],2), ",", round(filt_all_links$adj_p_BH[1],2), 
                           ",", round(filt_all_links$percent_transcribed_both[1]), "%), (", 
                           round(b2b_topall$pcc[1], 2), ",", round(b2b_topall$adj_p_BH[1],2), 
                           ",", round(b2b_topall$percent_transcribed_both[1]), "%), (",
                           round(sm_topall$pcc[1], 2), ",", round(sm_topall$adj_p_BH[1],2), 
                           ",", round(sm_topall$percent_transcribed_both[1]), "%)")
    
    top_b2b_value = paste0(top_b2b, "(", round(all_topb2b$pcc[1],2), ",", round(all_topb2b$adj_p_BH[1],2), 
                           ",", round(all_topb2b$percent_transcribed_both[1]), "%), (", 
                           round(filt_b2b_links$pcc[1], 2), ",", round(filt_b2b_links$adj_p_BH[1],2), 
                           ",", round(filt_b2b_links$percent_transcribed_both[1]), "%), (",
                           round(sm_topb2b$pcc[1], 2), ",", round(sm_topb2b$adj_p_BH[1],2), 
                           ",", round(sm_topb2b$percent_transcribed_both[1]), "%)")
    
    top_sm_value = paste0(top_sm, "(", round(all_topsm$pcc[1],2), ",", round(all_topsm$adj_p_BH[1],2), 
                           ",", round(all_topsm$percent_transcribed_both[1]), "%), (", 
                           round(b2b_topsm$pcc[1], 2), ",", round(b2b_topsm$adj_p_BH[1],2), 
                           ",", round(b2b_topsm$percent_transcribed_both[1]), "%), (",
                           round(filt_sm_links$pcc[1], 2), ",", round(filt_sm_links$adj_p_BH[1],2), 
                           ",", round(filt_sm_links$percent_transcribed_both[1]), "%)")
    
    top_disttss_value = paste0(top_disttss, "(", round(all_topdisttss$pcc[1],2), ",", round(all_topdisttss$adj_p_BH[1],2), 
                           ",", round(all_topdisttss$percent_transcribed_both[1]), "%), (", 
                           round(b2b_topdisttss$pcc[1], 2), ",", round(b2b_topdisttss$adj_p_BH[1],2), 
                           ",", round(b2b_topdisttss$percent_transcribed_both[1]), "%), (",
                           round(sm_topdisttss$pcc[1], 2), ",", round(sm_topdisttss$adj_p_BH[1],2), 
                           ",", round(sm_topdisttss$percent_transcribed_both[1]), "%)")
    
    top_disttes_value = paste0(top_disttes, "(", round(all_topdisttes$pcc[1],2), ",", round(all_topdisttes$adj_p_BH[1],2), 
                           ",", round(all_topdisttes$percent_transcribed_both[1]), "%), (", 
                           round(b2b_topdisttes$pcc[1], 2), ",", round(b2b_topdisttes$adj_p_BH[1],2), 
                           ",", round(b2b_topdisttes$percent_transcribed_both[1]), "%), (",
                           round(sm_topdisttes$pcc[1], 2), ",", round(sm_topdisttes$adj_p_BH[1],2), 
                           ",", round(sm_topdisttes$percent_transcribed_both[1]), "%)")

    return(c("Feature"=feat_use, "AllCorr"=top_all_value, "B2BCorr"=top_b2b_value, "smAECCorr"=top_sm_value, 
      "DistTSS"=top_disttss_value, "DistTES"=top_disttes_value))
    }

                       

In [34]:

length(intersect(mu_names, snps$central))



CHR,start,stop,central,note,BP,rsID,OR_copd,OR_asthma,P_copd,P_asthma,meta_p_weighted,same_direction
<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,966814,968936,chr1:968123,LIET Gene TSS Intron,967369,rs113967711,0.9846,0.9606,0.6414,0.02325,0.11765640,Yes
1,966814,968936,chr1:968123,LIET Gene TSS Intron,967941,rs6669800,1.0450,0.9737,0.1586,0.11470,0.05940026,No


## 2. Transform Counts
* Normalization
* Have 0 counts be replaced with NA (since not informative)

## 3. Save and run Ru's code?

In [ ]:
https://github.com/Dowell-Lab/bidir_gene_pairs/blob/windowed_correlations/R/nascent_correlations.R



